In [3]:
# ============================================================
# COMPLETE DENGUE DIFFERENTIAL-DIAGNOSIS IMPLEMENTATION
# Individual models + Borderline-SMOTE + stacking + XAI
# + exact mathematical reconstruction
# ============================================================

!pip install -q lime shap xgboost imbalanced-learn openpyxl

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from IPython.display import display
from lime.lime_tabular import LimeTabularExplainer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score,
    hamming_loss,
    log_loss,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

from sklearn.calibration import calibration_curve

from xgboost import XGBClassifier

from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully.")


# ============================================================
# 2. LOAD DATASET
# ============================================================

DATA_PATH = "/content/figsharecsv.csv"

df = pd.read_csv(DATA_PATH)

print("=" * 75)
print("DATASET INFORMATION")
print("=" * 75)

print("Dataset shape:", df.shape)
print("Number of participants:", df["id"].nunique())


# ============================================================
# 3. CREATE BINARY OUTCOME
# Dengue = 1
# Confirmed and probable/suspected CAB = 0
# ============================================================

target_col = (
    "dengue or probable (suspected) CABI(CAB) "
    "or confirmed CABI(CAB)"
)

print("\nOriginal diagnostic categories:")
print(df[target_col].value_counts(dropna=False))

df["is_dengue"] = (
    df[target_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.contains("dengue", na=False)
    .astype(int)
)

print("\nBinary observation distribution:")
print(df["is_dengue"].value_counts().sort_index())


# ============================================================
# 4. DATE CONVERSION AND TEMPORAL FEATURES
# ============================================================

date_columns = [
    "date of first symptom",
    "date of admission",
    "date of sampling"
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        format="%d-%m-%Y",
        errors="coerce"
    )

df["days_to_admission"] = (
    df["date of admission"] -
    df["date of first symptom"]
).dt.days

df["days_to_sampling"] = (
    df["date of sampling"] -
    df["date of admission"]
).dt.days


# ============================================================
# 5. SELECT FINAL PREDICTORS
# ============================================================

feature_columns = [
    "age",
    "sex",
    "days_to_admission",
    "days_to_sampling",
    "wbc",
    "neut%",
    "lymp%",
    "mono%",
    "eosino%",
    "rbc",
    "hb",
    "plt",
    "ipf%",
    "hct",
    "crp",
    "procalcitonin",
    "tbil",
    "ast",
    "alt",
    "bun",
    "creatinin"
]

df_model = df[
    feature_columns + ["is_dengue"]
].copy()

# Binary dummy encoding of sex
df_model = pd.get_dummies(
    df_model,
    columns=["sex"],
    drop_first=True,
    dtype=int
)

print("\nFinal columns before imputation:")
print(df_model.columns.tolist())

print("\nDataset shape before imputation:")
print(df_model.shape)


# ============================================================
# 6. MISSING-DATA TABLE
# ============================================================

missing_table = pd.DataFrame({
    "Variable": df_model.columns,
    "Missing_n": df_model.isna().sum().values,
    "Missing_percent": (
        df_model.isna().mean().values * 100
    )
})

missing_table = missing_table[
    missing_table["Variable"] != "is_dengue"
].copy()

missing_table["Missing_percent"] = (
    missing_table["Missing_percent"].round(2)
)

print("\nMissing-data table:")
display(missing_table)

missing_table.to_csv(
    "missing_data_table.csv",
    index=False
)

missing_table.to_excel(
    "missing_data_table.xlsx",
    index=False
)

# ============================================================
# 7. MEDIAN IMPUTATION
#
# ============================================================
# Descriptive statistics from original, non-imputed observations

descriptive_variables = [
    "age",
    "days_to_admission",
    "days_to_sampling",
    "wbc",
    "neut%",
    "lymp%",
    "mono%",
    "eosino%",
    "rbc",
    "hb",
    "plt",
    "ipf%",
    "hct",
    "crp",
    "procalcitonin",
    "tbil",
    "ast",
    "alt",
    "bun",
    "creatinin"
]

descriptive_df = df[
    descriptive_variables + ["is_dengue"]
].copy()

descriptive_rows = []

for variable in descriptive_variables:

    for class_value, class_name in [
        (1, "Dengue"),
        (0, "Non-dengue")
    ]:

        values = descriptive_df.loc[
            descriptive_df["is_dengue"] == class_value,
            variable
        ].dropna()

        descriptive_rows.append({
            "Variable": variable,
            "Group": class_name,
            "Valid n": len(values),
            "Mean": values.mean(),
            "Standard deviation": values.std(),
            "Median": values.median(),
            "25th percentile": values.quantile(0.25),
            "75th percentile": values.quantile(0.75)
        })

observation_descriptive_table = pd.DataFrame(
    descriptive_rows
)

numeric_columns = [
    "Mean",
    "Standard deviation",
    "Median",
    "25th percentile",
    "75th percentile"
]

observation_descriptive_table[numeric_columns] = (
    observation_descriptive_table[
        numeric_columns
    ].round(2)
)

display(observation_descriptive_table)

observation_descriptive_table.to_excel(
    "observation_level_descriptive_statistics.xlsx",
    index=False
)
#Generate descriptive-statistics table
# Improve names for manuscript presentation
variable_labels = {
    "age": "Age",
    "days_to_admission": "Days to admission",
    "days_to_sampling": "Days to sampling",
    "wbc": "White blood cell count",
    "neut%": "Neutrophil percentage",
    "lymp%": "Lymphocyte percentage",
    "mono%": "Monocyte percentage",
    "eosino%": "Eosinophil percentage",
    "rbc": "Red blood cell count",
    "hb": "Haemoglobin",
    "plt": "Platelet count",
    "ipf%": "Immature platelet fraction",
    "hct": "Haematocrit",
    "crp": "C-reactive protein",
    "procalcitonin": "Procalcitonin",
    "tbil": "Total bilirubin",
    "ast": "Aspartate aminotransferase",
    "alt": "Alanine aminotransferase",
    "bun": "Blood urea nitrogen",
    "creatinin": "Creatinine"
}

clean_descriptive_table = (
    observation_descriptive_table.copy()
)

clean_descriptive_table["Variable"] = (
    clean_descriptive_table["Variable"]
    .replace(variable_labels)
)

clean_descriptive_table = (
    clean_descriptive_table.rename(
        columns={
            "Standard deviation": "SD",
            "25th percentile": "Q1",
            "75th percentile": "Q3"
        }
    )
)

numeric_columns = [
    "Mean",
    "SD",
    "Median",
    "Q1",
    "Q3"
]

clean_descriptive_table[numeric_columns] = (
    clean_descriptive_table[numeric_columns]
    .round(2)
)

display(
    clean_descriptive_table.style.format({
        "Mean": "{:.2f}",
        "SD": "{:.2f}",
        "Median": "{:.2f}",
        "Q1": "{:.2f}",
        "Q3": "{:.2f}"
    })
)

clean_descriptive_table.to_excel(
    "supplementary_table_S1_descriptive_statistics.xlsx",
    index=False
)

clean_descriptive_table.to_csv(
    "supplementary_table_S1_descriptive_statistics.csv",
    index=False
)



predictor_columns = [
    column for column in df_model.columns
    if column != "is_dengue"
]

imputation_values = {}

for column in predictor_columns:
    median_value = df_model[column].median()
    imputation_values[column] = median_value

    df_model[column] = (
        df_model[column]
        .fillna(median_value)
    )

print("\nMissing values after imputation:")
print(df_model.isna().sum().sum())

print("Final analytical dataset shape:")
print(df_model.shape)


# ============================================================
# 8. COMMON TRAIN/TEST SPLIT FOR ALL MODELS
# Non-stratified split matching the older ensemble implementation
# ============================================================

X = df_model.drop(columns=["is_dengue"])
y = df_model["is_dengue"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("\n" + "=" * 75)
print("TRAINING AND TEST DISTRIBUTIONS")
print("=" * 75)

print("Training observations:", len(y_train))
print("Test observations:", len(y_test))

print("\nTraining distribution:")
print(y_train.value_counts().sort_index())

print("\nTest distribution:")
print(y_test.value_counts().sort_index())


# ============================================================
# 9. TABLE 4 HYPERPARAMETERS
# ============================================================

rf_parameters = {
    "bootstrap": True,
    "max_depth": 100,
    "max_features": 2,
    "min_samples_leaf": 4,
    "min_samples_split": 10,
    "n_estimators": 100,
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}

xgb_parameters = {
    "colsample_bytree": 1.0,
    "learning_rate": 0.1,
    "max_depth": 5,
    "n_estimators": 100,
    "subsample": 1.0,
    "eval_metric": "logloss",
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}

lr_parameters = {
    "C": 1,
    "penalty": "l2",
    "solver": "liblinear",
    "max_iter": 1000,
    "random_state": RANDOM_STATE
}

borderline_smote_parameters = {
    "sampling_strategy": "auto",
    "random_state": RANDOM_STATE,
    "k_neighbors": 5,
    "m_neighbors": 10,
    "kind": "borderline-1"
}


# ============================================================
# 10. DEFINE INDIVIDUAL MODEL PIPELINES
# ============================================================

rf_pipeline = ImbPipeline(steps=[
    ("scaler", StandardScaler()),
    (
        "borderline_smote",
        BorderlineSMOTE(
            **borderline_smote_parameters
        )
    ),
    (
        "classifier",
        RandomForestClassifier(
            **rf_parameters
        )
    )
])

xgb_pipeline = ImbPipeline(steps=[
    ("scaler", StandardScaler()),
    (
        "borderline_smote",
        BorderlineSMOTE(
            **borderline_smote_parameters
        )
    ),
    (
        "classifier",
        XGBClassifier(
            **xgb_parameters
        )
    )
])

lr_pipeline = ImbPipeline(steps=[
    ("scaler", StandardScaler()),
    (
        "borderline_smote",
        BorderlineSMOTE(
            **borderline_smote_parameters
        )
    ),
    (
        "classifier",
        LogisticRegression(
            **lr_parameters
        )
    )
])


# ============================================================
# 11. DEFINE STACKING ENSEMBLE
# ============================================================

stack_estimators = [
    (
        "rf",
        RandomForestClassifier(
            **rf_parameters
        )
    ),
    (
        "xgb",
        XGBClassifier(
            **xgb_parameters
        )
    ),
    (
        "lr",
        LogisticRegression(
            **lr_parameters
        )
    )
]

meta_model = LogisticRegression(
    **lr_parameters
)

stack_classifier = StackingClassifier(
    estimators=stack_estimators,
    final_estimator=meta_model,
    cv=5,
    stack_method="predict_proba",
    passthrough=True,
    n_jobs=-1
)

stack_pipeline = ImbPipeline(steps=[
    ("scaler", StandardScaler()),
    (
        "borderline_smote",
        BorderlineSMOTE(
            **borderline_smote_parameters
        )
    ),
    ("classifier", stack_classifier)
])

models = {
    "Random Forest": rf_pipeline,
    "XGBoost": xgb_pipeline,
    "Logistic Regression": lr_pipeline,
    "Stacked ensemble": stack_pipeline
}


# ============================================================
# 12. DISPLAY HYPERPARAMETERS
# ============================================================

hyperparameter_table = pd.DataFrame({
    "Classifier": [
        "Random Forest",
        "XGBoost",
        "Logistic Regression base learner",
        "Logistic Regression meta-learner"
    ],
    "Hyperparameters": [
        str({
            key: value for key, value in rf_parameters.items()
            if key not in ["random_state", "n_jobs"]
        }),
        str({
            key: value for key, value in xgb_parameters.items()
            if key not in [
                "random_state",
                "n_jobs",
                "eval_metric"
            ]
        }),
        str({
            key: value for key, value in lr_parameters.items()
            if key != "random_state"
        }),
        str({
            key: value for key, value in lr_parameters.items()
            if key != "random_state"
        })
    ]
})

print("\n" + "=" * 75)
print("TABLE 4 HYPERPARAMETERS")
print("=" * 75)

display(hyperparameter_table)

hyperparameter_table.to_excel(
    "table4_hyperparameters.xlsx",
    index=False
)


# ============================================================
# 13. TRAIN ALL FOUR MODELS
# ============================================================

print("\n" + "=" * 75)
print("MODEL TRAINING")
print("=" * 75)

for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)

print("All models trained successfully.")


# ============================================================
# 14. VERIFY BORDERLINE-SMOTE
# ============================================================

fitted_scaler_check = (
    stack_pipeline.named_steps["scaler"]
)

X_train_scaled_check = (
    fitted_scaler_check.transform(X_train)
)

borderline_check = BorderlineSMOTE(
    **borderline_smote_parameters
)

X_resampled_check, y_resampled_check = (
    borderline_check.fit_resample(
        X_train_scaled_check,
        y_train
    )
)

print("\nBefore Borderline-SMOTE:")
print(y_train.value_counts().sort_index())

print("\nAfter Borderline-SMOTE:")
print(
    pd.Series(
        y_resampled_check
    ).value_counts().sort_index()
)


# ============================================================
# 15. CONSISTENT MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model_name,
    model,
    X_evaluation,
    y_evaluation
):
    y_predicted = model.predict(X_evaluation)

    y_probability = model.predict_proba(
        X_evaluation
    )[:, 1]

    cm = confusion_matrix(
        y_evaluation,
        y_predicted,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(
        y_evaluation,
        y_predicted
    )

    precision = precision_score(
        y_evaluation,
        y_predicted,
        pos_label=1,
        average="binary",
        zero_division=0
    )

    sensitivity = recall_score(
        y_evaluation,
        y_predicted,
        pos_label=1,
        average="binary",
        zero_division=0
    )

    specificity = tn / (tn + fp)

    f1 = f1_score(
        y_evaluation,
        y_predicted,
        pos_label=1,
        average="binary",
        zero_division=0
    )

    jaccard = jaccard_score(
        y_evaluation,
        y_predicted,
        pos_label=1,
        average="binary",
        zero_division=0
    )

    hamming = hamming_loss(
        y_evaluation,
        y_predicted
    )

    probability_log_loss = log_loss(
        y_evaluation,
        y_probability,
        labels=[0, 1]
    )

    roc_auc = roc_auc_score(
        y_evaluation,
        y_probability
    )

    average_precision = average_precision_score(
        y_evaluation,
        y_probability
    )

    expected_jaccard = f1 / (2 - f1)

    if not np.isclose(
        jaccard,
        expected_jaccard,
        atol=1e-12
    ):
        raise ValueError(
            f"F1 and Jaccard mismatch for {model_name}"
        )

    return {
        "Algorithm": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": sensitivity,
        "Specificity": specificity,
        "F1-score": f1,
        "Hamming loss": hamming,
        "Jaccard score": jaccard,
        "Log loss": probability_log_loss,
        "ROC-AUC": roc_auc,
        "Average precision": average_precision,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


# ============================================================
# 16. GENERATE CORRECTED TABLE 5
# ============================================================

results = []

for model_name, model in models.items():
    results.append(
        evaluate_model(
            model_name,
            model,
            X_test,
            y_test
        )
    )

results_df = pd.DataFrame(results)

table5_columns = [
    "Algorithm",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Hamming loss",
    "Jaccard score",
    "Log loss"
]

table5 = results_df[
    table5_columns
].copy()

table5_numeric_columns = [
    column for column in table5.columns
    if column != "Algorithm"
]

table5[table5_numeric_columns] = (
    table5[table5_numeric_columns].round(4)
)

print("\n" + "=" * 75)
print("CORRECTED TABLE 5")
print("=" * 75)

display(table5)

table5.to_csv(
    "corrected_table5_borderline_smote.csv",
    index=False
)

table5.to_excel(
    "corrected_table5_borderline_smote.xlsx",
    index=False
)


# ============================================================
# 17. COMPLETE VERIFICATION TABLE
# ============================================================

verification_columns = [
    "Algorithm",
    "TN",
    "FP",
    "FN",
    "TP",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1-score",
    "Jaccard score",
    "Hamming loss",
    "ROC-AUC",
    "Average precision",
    "Log loss"
]

verification_table = results_df[
    verification_columns
].copy()

verification_numeric_columns = [
    column for column in verification_table.columns
    if column not in [
        "Algorithm",
        "TN",
        "FP",
        "FN",
        "TP"
    ]
]

verification_table[
    verification_numeric_columns
] = verification_table[
    verification_numeric_columns
].round(4)

print("\nComplete verification table:")
display(verification_table)

verification_table.to_excel(
    "complete_model_verification.xlsx",
    index=False
)


# ============================================================
# 18. CLASSIFICATION REPORTS
# ============================================================

for model_name, model in models.items():
    current_predictions = model.predict(X_test)

    print("\n" + "=" * 75)
    print(model_name)
    print("=" * 75)

    print(
        classification_report(
            y_test,
            current_predictions,
            target_names=[
                "Non-Dengue",
                "Dengue"
            ],
            digits=4
        )
    )


# ============================================================
# 19. ENSEMBLE CONFUSION MATRIX
# ============================================================

final_model = models["Stacked ensemble"]

y_pred_ensemble = final_model.predict(X_test)

y_prob_ensemble = final_model.predict_proba(
    X_test
)[:, 1]

cm_ensemble = confusion_matrix(
    y_test,
    y_pred_ensemble,
    labels=[0, 1]
)

cm_percentage = (
    cm_ensemble.astype(float) /
    cm_ensemble.sum(axis=1, keepdims=True)
) * 100

cm_annotations = np.empty_like(
    cm_ensemble,
    dtype=object
)

for row in range(cm_ensemble.shape[0]):
    for column in range(cm_ensemble.shape[1]):
        cm_annotations[row, column] = (
            f"{cm_ensemble[row, column]}\n"
            f"({cm_percentage[row, column]:.2f}%)"
        )

plt.figure(figsize=(7, 5.5))

sns.heatmap(
    cm_ensemble,
    annot=cm_annotations,
    fmt="",
    cmap="Blues",
    square=True,
    linewidths=0.8,
    linecolor="white",
    xticklabels=[
        "Non-Dengue",
        "Dengue"
    ],
    yticklabels=[
        "Non-Dengue",
        "Dengue"
    ],
    annot_kws={"size": 13}
)

plt.title(
    "Confusion Matrix – Stacked Ensemble Model",
    fontsize=14,
    pad=15
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    "stacked_ensemble_confusion_matrix.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Ensemble confusion matrix:")
print(cm_ensemble)

#new confidence-interval and calibration cell here.

# ============================================================
# ADDITIONAL STATISTICAL EVALUATION FOR REVIEWER RESPONSE
# 95% bootstrap confidence intervals and calibration statistics
#
# INSERT AFTER:
#   rf_pipeline.fit(...)
#   xgb_pipeline.fit(...)
#   lr_pipeline.fit(...)
#   stack_pipeline.fit(...)
#
# INSERT BEFORE:
#   SHAP and LIME analysis
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# 1. Specify the fitted model objects
# Change these names only if different names were used earlier
# ------------------------------------------------------------

MODELS = {
    "Random Forest": rf_pipeline,
    "XGBoost": xgb_pipeline,
    "Logistic Regression": lr_pipeline,
    "Stacked ensemble": stack_pipeline
}

# X_test and y_test must be exactly the same unchanged test set
# used for Table 8 and the confusion matrices.
y_test_array = np.asarray(y_test).astype(int)

print("Test-set size:", len(y_test_array))
print("Non-dengue observations:", np.sum(y_test_array == 0))
print("Dengue observations:", np.sum(y_test_array == 1))

# Expected:
# Total = 397
# Non-dengue = 285
# Dengue = 112

# ------------------------------------------------------------
# 2. Function for point estimates
# ------------------------------------------------------------

def calculate_metrics(y_true, y_probability, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    y_probability = np.asarray(y_probability, dtype=float)

    y_prediction = (y_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_prediction, labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_prediction),
        "Precision": precision_score(
            y_true, y_prediction, zero_division=0
        ),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1-score": f1_score(
            y_true, y_prediction, zero_division=0
        ),
        "AUROC": roc_auc_score(y_true, y_probability),
        "AUPRC": average_precision_score(y_true, y_probability),
        "Brier score": brier_score_loss(y_true, y_probability)
    }

# ------------------------------------------------------------
# 3. Calibration intercept and slope
#
# Logistic calibration model:
# logit(observed outcome) =
# calibration intercept +
# calibration slope × logit(predicted probability)
#
# Ideal values:
# intercept = 0
# slope = 1
# ------------------------------------------------------------

def calculate_calibration_statistics(y_true, y_probability):
    y_true = np.asarray(y_true).astype(int)
    y_probability = np.asarray(y_probability, dtype=float)

    # Avoid log(0) and division by zero
    epsilon = 1e-6
    probability_clipped = np.clip(
        y_probability, epsilon, 1 - epsilon
    )

    predicted_logit = np.log(
        probability_clipped / (1 - probability_clipped)
    ).reshape(-1, 1)

    # A very large C approximates an unpenalized logistic model
    calibration_model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=10000
    )

    calibration_model.fit(predicted_logit, y_true)

    calibration_intercept = calibration_model.intercept_[0]
    calibration_slope = calibration_model.coef_[0][0]

    return calibration_intercept, calibration_slope

# ------------------------------------------------------------
# 4. Observation-level bootstrap confidence intervals
#
# The bootstrap is stratified by outcome so every resample
# contains both dengue and non-dengue observations.
# ------------------------------------------------------------

def stratified_bootstrap_confidence_intervals(
    y_true,
    y_probability,
    n_bootstrap=2000,
    confidence_level=0.95,
    random_state=42
):
    y_true = np.asarray(y_true).astype(int)
    y_probability = np.asarray(y_probability, dtype=float)

    rng = np.random.default_rng(random_state)

    negative_indices = np.where(y_true == 0)[0]
    positive_indices = np.where(y_true == 1)[0]

    bootstrap_results = []

    for _ in range(n_bootstrap):
        sampled_negative = rng.choice(
            negative_indices,
            size=len(negative_indices),
            replace=True
        )

        sampled_positive = rng.choice(
            positive_indices,
            size=len(positive_indices),
            replace=True
        )

        sampled_indices = np.concatenate(
            [sampled_negative, sampled_positive]
        )

        # Shuffle the combined bootstrap sample
        rng.shuffle(sampled_indices)

        sampled_y = y_true[sampled_indices]
        sampled_probability = y_probability[sampled_indices]

        metrics = calculate_metrics(
            sampled_y,
            sampled_probability,
            threshold=0.50
        )

        bootstrap_results.append(metrics)

    bootstrap_df = pd.DataFrame(bootstrap_results)

    alpha = 1 - confidence_level
    lower_percentile = 100 * alpha / 2
    upper_percentile = 100 * (1 - alpha / 2)

    intervals = {}

    for metric in bootstrap_df.columns:
        intervals[metric] = (
            np.percentile(
                bootstrap_df[metric].dropna(),
                lower_percentile
            ),
            np.percentile(
                bootstrap_df[metric].dropna(),
                upper_percentile
            )
        )

    return intervals

#Run the NPV and NPV-confidence-interval
# ============================================================
# NPV WITH 95% STRATIFIED BOOTSTRAP CONFIDENCE INTERVAL
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# Use the fitted model names from the previous evaluation code
MODELS = {
    "Random Forest": rf_pipeline,
    "XGBoost": xgb_pipeline,
    "Logistic Regression": lr_pipeline,
    "Stacked ensemble": stack_pipeline
}

y_true = np.asarray(y_test).astype(int)

def calculate_npv(y_true, y_probability, threshold=0.50):
    """Calculate negative predictive value."""

    y_prediction = (
        np.asarray(y_probability) >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_prediction,
        labels=[0, 1]
    ).ravel()

    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

    return npv, tn, fp, fn, tp


def bootstrap_npv_ci(
    y_true,
    y_probability,
    n_bootstrap=2000,
    confidence_level=0.95,
    random_state=42
):
    """Calculate a stratified observation-level bootstrap CI for NPV."""

    y_true = np.asarray(y_true).astype(int)
    y_probability = np.asarray(y_probability, dtype=float)

    rng = np.random.default_rng(random_state)

    negative_indices = np.where(y_true == 0)[0]
    positive_indices = np.where(y_true == 1)[0]

    bootstrap_values = []

    for _ in range(n_bootstrap):

        sampled_negative = rng.choice(
            negative_indices,
            size=len(negative_indices),
            replace=True
        )

        sampled_positive = rng.choice(
            positive_indices,
            size=len(positive_indices),
            replace=True
        )

        sampled_indices = np.concatenate(
            [sampled_negative, sampled_positive]
        )

        rng.shuffle(sampled_indices)

        sampled_y = y_true[sampled_indices]
        sampled_probability = y_probability[sampled_indices]

        sampled_npv, _, _, _, _ = calculate_npv(
            sampled_y,
            sampled_probability,
            threshold=0.50
        )

        if not np.isnan(sampled_npv):
            bootstrap_values.append(sampled_npv)

    alpha = 1 - confidence_level

    lower = np.percentile(
        bootstrap_values,
        100 * alpha / 2
    )

    upper = np.percentile(
        bootstrap_values,
        100 * (1 - alpha / 2)
    )

    return lower, upper


# Evaluate all models
npv_results = []

for model_number, (model_name, fitted_model) in enumerate(
    MODELS.items()
):
    y_probability = fitted_model.predict_proba(X_test)[:, 1]

    npv, tn, fp, fn, tp = calculate_npv(
        y_true,
        y_probability,
        threshold=0.50
    )

    lower_ci, upper_ci = bootstrap_npv_ci(
        y_true,
        y_probability,
        n_bootstrap=2000,
        confidence_level=0.95,
        random_state=42 + model_number
    )

    npv_results.append({
        "Model": model_name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "NPV": npv,
        "NPV 95% CI lower": lower_ci,
        "NPV 95% CI upper": upper_ci,
        "NPV (95% CI)": (
            f"{npv:.4f} "
            f"({lower_ci:.4f}–{upper_ci:.4f})"
        )
    })

npv_table = pd.DataFrame(npv_results)

print("\nNPV RESULTS")
print(npv_table.to_string(index=False))

npv_table.to_csv(
    "npv_with_95_percent_bootstrap_ci.csv",
    index=False
)

print("\nSaved: npv_with_95_percent_bootstrap_ci.csv")

# ------------------------------------------------------------
# 5. Evaluate every fitted model
# ------------------------------------------------------------

summary_rows = []
formatted_rows = []
prediction_storage = {}

for model_number, (model_name, fitted_model) in enumerate(
    MODELS.items()
):
    print("\nEvaluating:", model_name)

    probability = fitted_model.predict_proba(X_test)[:, 1]
    prediction = (probability >= 0.50).astype(int)

    prediction_storage[model_name] = {
        "probability": probability,
        "prediction": prediction
    }

    point_estimates = calculate_metrics(
        y_test_array,
        probability,
        threshold=0.50
    )

    calibration_intercept, calibration_slope = (
        calculate_calibration_statistics(
            y_test_array,
            probability
        )
    )

    confidence_intervals = (
        stratified_bootstrap_confidence_intervals(
            y_test_array,
            probability,
            n_bootstrap=2000,
            confidence_level=0.95,
            random_state=42 + model_number
        )
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test_array,
        prediction,
        labels=[0, 1]
    ).ravel()

    summary_row = {
        "Model": model_name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        **point_estimates,
        "Calibration intercept": calibration_intercept,
        "Calibration slope": calibration_slope
    }

    # Add lower and upper confidence limits separately
    for metric, (lower, upper) in confidence_intervals.items():
        summary_row[f"{metric} CI lower"] = lower
        summary_row[f"{metric} CI upper"] = upper

    summary_rows.append(summary_row)

    # Create publication-ready formatted values
    formatted_row = {
        "Model": model_name,
        "Accuracy (95% CI)": (
            f"{point_estimates['Accuracy']:.4f} "
            f"({confidence_intervals['Accuracy'][0]:.4f}–"
            f"{confidence_intervals['Accuracy'][1]:.4f})"
        ),
        "Precision (95% CI)": (
            f"{point_estimates['Precision']:.4f} "
            f"({confidence_intervals['Precision'][0]:.4f}–"
            f"{confidence_intervals['Precision'][1]:.4f})"
        ),
        "Sensitivity (95% CI)": (
            f"{point_estimates['Sensitivity']:.4f} "
            f"({confidence_intervals['Sensitivity'][0]:.4f}–"
            f"{confidence_intervals['Sensitivity'][1]:.4f})"
        ),
        "Specificity (95% CI)": (
            f"{point_estimates['Specificity']:.4f} "
            f"({confidence_intervals['Specificity'][0]:.4f}–"
            f"{confidence_intervals['Specificity'][1]:.4f})"
        ),
        "F1-score (95% CI)": (
            f"{point_estimates['F1-score']:.4f} "
            f"({confidence_intervals['F1-score'][0]:.4f}–"
            f"{confidence_intervals['F1-score'][1]:.4f})"
        ),
        "AUROC (95% CI)": (
            f"{point_estimates['AUROC']:.4f} "
            f"({confidence_intervals['AUROC'][0]:.4f}–"
            f"{confidence_intervals['AUROC'][1]:.4f})"
        ),
        "AUPRC (95% CI)": (
            f"{point_estimates['AUPRC']:.4f} "
            f"({confidence_intervals['AUPRC'][0]:.4f}–"
            f"{confidence_intervals['AUPRC'][1]:.4f})"
        ),
        "Brier score": f"{point_estimates['Brier score']:.4f}",
        "Calibration intercept": f"{calibration_intercept:.4f}",
        "Calibration slope": f"{calibration_slope:.4f}"
    }

    formatted_rows.append(formatted_row)

# ------------------------------------------------------------
# 6. Display the results
# ------------------------------------------------------------

complete_results = pd.DataFrame(summary_rows)
publication_table = pd.DataFrame(formatted_rows)

print("\n" + "=" * 100)
print("COMPLETE NUMERICAL RESULTS")
print("=" * 100)
print(complete_results.round(4).to_string(index=False))

print("\n" + "=" * 100)
print("PUBLICATION-READY TABLE")
print("=" * 100)
print(publication_table.to_string(index=False))

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

complete_results.to_csv(
    "complete_metrics_with_bootstrap_CI.csv",
    index=False
)

publication_table.to_csv(
    "publication_calibration_table.csv",
    index=False
)

print("\nFiles saved:")
print("1. complete_metrics_with_bootstrap_CI.csv")
print("2. publication_calibration_table.csv")
# ============================================================
# 20. ROC AND PRECISION-RECALL CURVES
# ============================================================

fpr, tpr, _ = roc_curve(
    y_test,
    y_prob_ensemble,
    pos_label=1
)

ensemble_roc_auc = roc_auc_score(
    y_test,
    y_prob_ensemble
)

pr_precision, pr_recall, _ = (
    precision_recall_curve(
        y_test,
        y_prob_ensemble,
        pos_label=1
    )
)

ensemble_average_precision = (
    average_precision_score(
        y_test,
        y_prob_ensemble
    )
)

positive_prevalence = (
    y_test == 1
).mean()

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5.5)
)

axes[0].plot(
    fpr,
    tpr,
    color="darkblue",
    linewidth=2.5,
    label=f"AUC = {ensemble_roc_auc:.4f}"
)

axes[0].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Random classifier"
)

axes[0].set_xlabel(
    "False Positive Rate (1 − Specificity)"
)
axes[0].set_ylabel(
    "True Positive Rate (Sensitivity)"
)
axes[0].set_title("(a) ROC Curve")
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])
axes[0].grid(alpha=0.25)
axes[0].legend(loc="lower right")

axes[1].plot(
    pr_recall,
    pr_precision,
    color="darkred",
    linewidth=2.5,
    label=(
        f"Average precision = "
        f"{ensemble_average_precision:.4f}"
    )
)

axes[1].axhline(
    y=positive_prevalence,
    linestyle="--",
    color="gray",
    label=(
        f"No-skill baseline = "
        f"{positive_prevalence:.4f}"
    )
)

axes[1].set_xlabel("Recall (Sensitivity)")
axes[1].set_ylabel("Precision")
axes[1].set_title("(b) Precision–Recall Curve")
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])
axes[1].grid(alpha=0.25)
axes[1].legend(loc="lower left")

fig.suptitle(
    "Performance Curves of the Stacked Ensemble",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    "stacked_ensemble_roc_pr_curves.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 21. EXTRACT META-MODEL COEFFICIENTS WITH FEATURE NAMES
# ============================================================

fitted_scaler = (
    stack_pipeline.named_steps["scaler"]
)

fitted_stack = (
    stack_pipeline.named_steps["classifier"]
)

fitted_meta_model = (
    fitted_stack.final_estimator_
)

meta_coefficients = (
    fitted_meta_model.coef_.ravel()
)

meta_intercept = float(
    fitted_meta_model.intercept_[0]
)

base_output_names = [
    "RF predicted dengue probability",
    "XGBoost predicted dengue probability",
    "LR predicted dengue probability"
]

original_feature_names = [
    f"Standardized {feature}"
    for feature in X.columns.tolist()
]

meta_feature_names = (
    base_output_names +
    original_feature_names
)

if len(meta_coefficients) != len(meta_feature_names):
    raise ValueError(
        "Meta-feature names and coefficients do not match."
    )

meta_coefficient_table = pd.DataFrame({
    "Meta-feature": meta_feature_names,
    "Coefficient": meta_coefficients,
    "Absolute coefficient": np.abs(
        meta_coefficients
    )
})

meta_coefficient_table["Direction"] = np.where(
    meta_coefficient_table["Coefficient"] > 0,
    "Toward dengue",
    np.where(
        meta_coefficient_table["Coefficient"] < 0,
        "Toward non-dengue",
        "No contribution"
    )
)

intercept_row = pd.DataFrame({
    "Meta-feature": ["Intercept"],
    "Coefficient": [meta_intercept],
    "Absolute coefficient": [
        abs(meta_intercept)
    ],
    "Direction": [
        "Baseline decision score"
    ]
})

complete_coefficient_table = pd.concat(
    [
        intercept_row,
        meta_coefficient_table
    ],
    ignore_index=True
)

print("\n" + "=" * 75)
print("META-MODEL COEFFICIENTS")
print("=" * 75)

display(complete_coefficient_table)

complete_coefficient_table.to_excel(
    "table6_meta_model_coefficients.xlsx",
    index=False
)


# ============================================================
# 22. EXACT MATHEMATICAL RECONSTRUCTION
# ============================================================

# The stack receives standardized test data.
X_test_scaled_stack = fitted_scaler.transform(
    X_test
)

# Retrieve fitted base learners from inside the stack.
fitted_rf_base = (
    fitted_stack.named_estimators_["rf"]
)

fitted_xgb_base = (
    fitted_stack.named_estimators_["xgb"]
)

fitted_lr_base = (
    fitted_stack.named_estimators_["lr"]
)

rf_base_probability = (
    fitted_rf_base.predict_proba(
        X_test_scaled_stack
    )[:, 1]
)

xgb_base_probability = (
    fitted_xgb_base.predict_proba(
        X_test_scaled_stack
    )[:, 1]
)

lr_base_probability = (
    fitted_lr_base.predict_proba(
        X_test_scaled_stack
    )[:, 1]
)

# Because passthrough=True, append standardized original features.
manual_meta_features = np.column_stack([
    rf_base_probability,
    xgb_base_probability,
    lr_base_probability,
    X_test_scaled_stack
])

if manual_meta_features.shape[1] != len(
    meta_coefficients
):
    raise ValueError(
        "Manual meta-feature count does not match "
        "the meta-model coefficient count."
    )

# Explicit linear equation
mathematical_linear_score = (
    meta_intercept +
    manual_meta_features @ meta_coefficients
)

# Sigmoid transformation
mathematical_probability = (
    1 /
    (
        1 +
        np.exp(
            -mathematical_linear_score
        )
    )
)

mathematical_prediction = (
    mathematical_probability >= 0.50
).astype(int)

# Compare with StackingClassifier output
maximum_probability_difference = np.max(
    np.abs(
        y_prob_ensemble -
        mathematical_probability
    )
)

prediction_agreement = np.mean(
    y_pred_ensemble ==
    mathematical_prediction
)

print("\n" + "=" * 75)
print("MATHEMATICAL RECONSTRUCTION VERIFICATION")
print("=" * 75)

print(
    "Maximum probability difference:",
    f"{maximum_probability_difference:.12f}"
)

print(
    "Prediction agreement:",
    f"{prediction_agreement:.4f}"
)

if maximum_probability_difference < 1e-10:
    print(
        "Exact mathematical reconstruction confirmed."
    )
else:
    print(
        "Warning: mathematical probabilities do not "
        "exactly reproduce the stacking output."
    )


# ============================================================
# 23. EVALUATE MATHEMATICAL MODEL
# ============================================================

mathematical_accuracy = accuracy_score(
    y_test,
    mathematical_prediction
)

mathematical_precision = precision_score(
    y_test,
    mathematical_prediction,
    pos_label=1,
    zero_division=0
)

mathematical_recall = recall_score(
    y_test,
    mathematical_prediction,
    pos_label=1,
    zero_division=0
)

mathematical_f1 = f1_score(
    y_test,
    mathematical_prediction,
    pos_label=1,
    zero_division=0
)

mathematical_auc = roc_auc_score(
    y_test,
    mathematical_probability
)

print("\nMathematical model results:")
print(
    f"Accuracy:  {mathematical_accuracy:.4f}"
)
print(
    f"Precision: {mathematical_precision:.4f}"
)
print(
    f"Recall:    {mathematical_recall:.4f}"
)
print(
    f"F1-score:  {mathematical_f1:.4f}"
)
print(
    f"ROC-AUC:   {mathematical_auc:.4f}"
)


# ============================================================
# 24. ROC COMPARISON
# ============================================================

fpr_math, tpr_math, _ = roc_curve(
    y_test,
    mathematical_probability
)

plt.figure(figsize=(7, 6))

plt.plot(
    fpr,
    tpr,
    linewidth=2.5,
    label=(
        "Stacking ensemble "
        f"(AUC = {ensemble_roc_auc:.4f})"
    )
)

plt.plot(
    fpr_math,
    tpr_math,
    linestyle="--",
    linewidth=2,
    label=(
        "Mathematical reconstruction "
        f"(AUC = {mathematical_auc:.4f})"
    )
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle=":",
    color="gray"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(
    "ROC Comparison of Stacking and "
    "Mathematical Reconstruction"
)
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "stacking_mathematical_roc_comparison.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 25. CALIBRATION COMPARISON
# ============================================================

ensemble_fraction_positive, ensemble_mean_predicted = (
    calibration_curve(
        y_test,
        y_prob_ensemble,
        n_bins=10,
        strategy="uniform"
    )
)

math_fraction_positive, math_mean_predicted = (
    calibration_curve(
        y_test,
        mathematical_probability,
        n_bins=10,
        strategy="uniform"
    )
)

plt.figure(figsize=(7, 6))

plt.plot(
    ensemble_mean_predicted,
    ensemble_fraction_positive,
    marker="o",
    label="Stacking ensemble"
)

plt.plot(
    math_mean_predicted,
    math_fraction_positive,
    marker="s",
    linestyle="--",
    label="Mathematical reconstruction"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle=":",
    color="black",
    label="Perfect calibration"
)

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed dengue proportion")
plt.title("Calibration Comparison")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "stacking_mathematical_calibration.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 26. PROBABILITY AGREEMENT SCATTER PLOT
# ============================================================

plt.figure(figsize=(7, 6))

plt.scatter(
    y_prob_ensemble,
    mathematical_probability,
    alpha=0.65,
    color="purple"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="black"
)

plt.xlabel("Stacking predicted probability")
plt.ylabel("Mathematical predicted probability")
plt.title(
    "Agreement Between Stacking and "
    "Mathematical Probabilities"
)
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "stacking_mathematical_probability_agreement.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 27. SIGMOID DECISION FUNCTION
# ============================================================

z_values = np.linspace(-10, 10, 500)

sigmoid_values = (
    1 /
    (
        1 +
        np.exp(-z_values)
    )
)

plt.figure(figsize=(7, 5))

plt.plot(
    z_values,
    sigmoid_values,
    color="darkblue",
    linewidth=2.5
)

plt.axhline(
    0.5,
    linestyle="--",
    color="gray"
)

plt.axvline(
    0,
    linestyle="--",
    color="gray"
)

plt.xlabel("Linear decision score z")
plt.ylabel("Predicted dengue probability")
plt.title("Logistic Sigmoid Decision Function")
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "mathematical_model_sigmoid.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 28. MATHEMATICAL PROBABILITY DISTRIBUTION
# ============================================================

probability_distribution = pd.DataFrame({
    "Probability": mathematical_probability,
    "True class": np.where(
        y_test.to_numpy() == 1,
        "Dengue",
        "Non-Dengue"
    )
})

plt.figure(figsize=(8, 5.5))

sns.histplot(
    data=probability_distribution,
    x="Probability",
    hue="True class",
    bins=20,
    stat="density",
    common_norm=False,
    element="step",
    alpha=0.35
)

plt.xlabel("Predicted dengue probability")
plt.ylabel("Density")
plt.title(
    "Probability Distribution of the "
    "Mathematical Reconstruction"
)
plt.tight_layout()

plt.savefig(
    "mathematical_probability_distribution.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 29. SHAP FOR ACTUAL XGBOOST BASE LEARNER INSIDE STACK
# ============================================================

X_test_scaled_df = pd.DataFrame(
    X_test_scaled_stack,
    columns=X.columns,
    index=X_test.index
)

shap_explainer = shap.TreeExplainer(
    fitted_xgb_base
)

shap_values = shap_explainer(
    X_test_scaled_df
)

# SHAP beeswarm
shap.summary_plot(
    shap_values.values,
    X_test_scaled_df,
    feature_names=X.columns,
    show=False
)

plt.title(
    "SHAP Summary Plot – XGBoost Base Learner"
)
plt.tight_layout()

plt.savefig(
    "xgboost_shap_summary.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Global SHAP importance
mean_absolute_shap = np.abs(
    shap_values.values
).mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": X.columns,
    "Mean absolute SHAP value":
        mean_absolute_shap
}).sort_values(
    "Mean absolute SHAP value",
    ascending=False
).reset_index(drop=True)

print("\nGlobal SHAP importance:")
display(shap_importance)

shap_importance.to_excel(
    "xgboost_shap_importance.xlsx",
    index=False
)

top_shap = (
    shap_importance
    .head(15)
    .sort_values(
        "Mean absolute SHAP value"
    )
)

plt.figure(figsize=(8, 7))

plt.barh(
    top_shap["Feature"],
    top_shap[
        "Mean absolute SHAP value"
    ],
    color="steelblue"
)

plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Feature")
plt.title(
    "Global SHAP Importance – XGBoost Base Learner"
)
plt.tight_layout()

plt.savefig(
    "xgboost_global_shap_importance.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 30. LIME FOR ACTUAL XGBOOST BASE LEARNER INSIDE STACK
# ============================================================

X_train_scaled_stack = fitted_scaler.transform(
    X_train
)

X_train_scaled_df = pd.DataFrame(
    X_train_scaled_stack,
    columns=X.columns,
    index=X_train.index
)

# Predictions from actual fitted XGBoost learner inside stack
y_pred_xgb_base = fitted_xgb_base.predict(
    X_test_scaled_stack
)

y_prob_xgb_base = (
    fitted_xgb_base.predict_proba(
        X_test_scaled_stack
    )[:, 1]
)

lime_prediction_table = pd.DataFrame({
    "True_label": y_test.to_numpy(),
    "Predicted_label": y_pred_xgb_base,
    "Dengue_probability": y_prob_xgb_base
}, index=X_test.index)

lime_explainer = LimeTabularExplainer(
    training_data=X_train_scaled_df.to_numpy(),
    feature_names=X.columns.tolist(),
    class_names=[
        "not_dengue",
        "dengue"
    ],
    mode="classification",
    discretize_continuous=True,
    random_state=RANDOM_STATE
)

# Select true-positive cases nearest 0.80 and 0.98
correct_dengue_cases = lime_prediction_table[
    (lime_prediction_table["True_label"] == 1) &
    (
        lime_prediction_table[
            "Predicted_label"
        ] == 1
    )
].copy()

if correct_dengue_cases.empty:
    raise ValueError(
        "No correctly classified dengue observations "
        "were available for LIME."
    )

target_probabilities = [0.80, 0.98]
selected_cases = {}

for target_probability in target_probabilities:
    probability_difference = np.abs(
        correct_dengue_cases[
            "Dengue_probability"
        ] - target_probability
    )

    available_difference = (
        probability_difference.drop(
            labels=list(
                selected_cases.values()
            ),
            errors="ignore"
        )
    )

    if available_difference.empty:
        available_difference = (
            probability_difference
        )

    selected_index = (
        available_difference.idxmin()
    )

    selected_cases[
        target_probability
    ] = selected_index

lime_summary_rows = []

for target_probability, selected_index in (
    selected_cases.items()
):
    test_position = X_test.index.get_loc(
        selected_index
    )

    selected_observation = (
        X_test_scaled_df.iloc[
            test_position
        ].to_numpy()
    )

    lime_explanation = (
        lime_explainer.explain_instance(
            data_row=selected_observation,
            predict_fn=(
                fitted_xgb_base.predict_proba
            ),
            labels=(1,),
            num_features=min(
                10,
                X.shape[1]
            ),
            num_samples=5000
        )
    )

    actual_probability = float(
        y_prob_xgb_base[test_position]
    )

    true_class = int(
        y_test.iloc[test_position]
    )

    predicted_class = int(
        y_pred_xgb_base[test_position]
    )

    print("\n" + "=" * 75)
    print(
        f"LIME EXPLANATION CLOSE TO "
        f"{target_probability:.2f}"
    )
    print("=" * 75)

    print("Dataset index:", selected_index)
    print("True class:", true_class)
    print("Predicted class:", predicted_class)
    print(
        "Actual dengue probability:",
        f"{actual_probability:.4f}"
    )

    print("\nLIME contributions:")

    for condition, weight in (
        lime_explanation.as_list(
            label=1
        )
    ):
        print(
            f"{condition}: {weight:.6f}"
        )

    # Standard three-panel LIME output
    lime_explanation.show_in_notebook(
        labels=(1,),
        show_table=True,
        show_all=False
    )

    probability_filename = (
        f"{actual_probability:.4f}"
        .replace(".", "_")
    )

    lime_explanation.save_to_file(
        f"lime_probability_"
        f"{probability_filename}.html"
    )

    lime_figure = (
        lime_explanation.as_pyplot_figure(
            label=1
        )
    )

    plt.title(
        "LIME Explanation – XGBoost Base Learner\n"
        f"Dengue probability = "
        f"{actual_probability:.4f}"
    )

    plt.tight_layout()

    plt.savefig(
        f"lime_probability_"
        f"{probability_filename}.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()

    lime_summary_rows.append({
        "Selection target": target_probability,
        "Dataset index": selected_index,
        "True class": true_class,
        "Predicted class": predicted_class,
        "Actual dengue probability":
            actual_probability
    })

lime_summary = pd.DataFrame(
    lime_summary_rows
)

lime_summary[
    "Actual dengue probability"
] = lime_summary[
    "Actual dengue probability"
].round(4)

print("\nSelected LIME cases:")
display(lime_summary)

lime_summary.to_excel(
    "lime_selected_cases.xlsx",
    index=False
)


# ============================================================
# 31. SAVE FINAL PREDICTIONS
# ============================================================

final_prediction_output = pd.DataFrame({
    "Original row index": X_test.index,
    "True class": y_test.to_numpy(),
    "Ensemble predicted class":
        y_pred_ensemble,
    "Ensemble dengue probability":
        y_prob_ensemble,
    "Mathematical predicted class":
        mathematical_prediction,
    "Mathematical dengue probability":
        mathematical_probability,
    "XGBoost base predicted class":
        y_pred_xgb_base,
    "XGBoost base dengue probability":
        y_prob_xgb_base
})

final_prediction_output.to_csv(
    "final_test_predictions.csv",
    index=False
)

final_prediction_output.to_excel(
    "final_test_predictions.xlsx",
    index=False
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

ensemble_result = results_df[
    results_df["Algorithm"] ==
    "Stacked ensemble"
].iloc[0]

print("\n" + "=" * 75)
print("FINAL STACKED ENSEMBLE SUMMARY")
print("=" * 75)

print(
    f"Accuracy:          "
    f"{ensemble_result['Accuracy']:.4f}"
)

print(
    f"Precision:         "
    f"{ensemble_result['Precision']:.4f}"
)

print(
    f"Recall:            "
    f"{ensemble_result['Recall']:.4f}"
)

print(
    f"Specificity:       "
    f"{ensemble_result['Specificity']:.4f}"
)

print(
    f"F1-score:          "
    f"{ensemble_result['F1-score']:.4f}"
)

print(
    f"Jaccard score:     "
    f"{ensemble_result['Jaccard score']:.4f}"
)

print(
    f"Hamming loss:      "
    f"{ensemble_result['Hamming loss']:.4f}"
)

print(
    f"Log loss:          "
    f"{ensemble_result['Log loss']:.4f}"
)

print(
    f"ROC-AUC:           "
    f"{ensemble_result['ROC-AUC']:.4f}"
)

print(
    f"Average precision: "
    f"{ensemble_result['Average precision']:.4f}"
)

print("\nMathematical reconstruction:")
print(
    f"Maximum probability difference: "
    f"{maximum_probability_difference:.12f}"
)

print(
    f"Prediction agreement: "
    f"{prediction_agreement:.4f}"
)

print("\nAnalysis completed successfully.")
print(
    "Use only the results generated by this run "
    "in Tables 5–6 and Figures 9–21."
)

Output hidden; open in https://colab.research.google.com to view.